<a href="https://colab.research.google.com/github/EulerQuant/data-analyst-portfolio/blob/main/FIFA_2026_Prediction_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ── UPLOAD FILES ──
uploaded = files.upload()  # Upload both CSVs when prompted

filenames = list(uploaded.keys())
df1 = pd.read_csv(filenames[0])
df2 = pd.read_csv(filenames[1])

# ── FIX TEAM NAMES ──
df1['team'] = df1['team'].replace({'Turkey': 'Türkiye', 'United States': 'USA'})
df2_groups = df2[df2['stage'] == 'Group Stage'].copy()

# ── TEAM STATS ──
team_stats = df1.groupby('team').agg(
    avg_goals=('goals', 'mean'),
    avg_xg=('expected_goals_xg', 'mean'),
    avg_player_rating=('player_rating', 'mean'),
    avg_defensive_actions=('defensive_actions', 'mean'),
    win_rate=('match_result', lambda x: (x == 'W').mean()),
    draw_rate=('match_result', lambda x: (x == 'D').mean()),
    loss_rate=('match_result', lambda x: (x == 'L').mean()),
).reset_index()

# ── MERGE ──
df_matches = df2_groups.copy()
df_matches = df_matches.merge(team_stats, left_on='team1', right_on='team', how='left').drop('team', axis=1)
df_matches.columns = [c if c not in team_stats.columns[1:] else 'team1_' + c for c in df_matches.columns]
df_matches = df_matches.merge(team_stats, left_on='team2', right_on='team', how='left').drop('team', axis=1)
df_matches.columns = [c if c not in team_stats.columns[1:] else 'team2_' + c for c in df_matches.columns]

for col in team_stats.columns[1:]:
    df_matches[f'team1_{col}'] = df_matches[f'team1_{col}'].fillna(team_stats[col].mean())
    df_matches[f'team2_{col}'] = df_matches[f'team2_{col}'].fillna(team_stats[col].mean())

df_matches['fifa_rank_diff'] = df_matches['team1_fifa_rank'] - df_matches['team2_fifa_rank']

# ── PREDICT WINNER FUNCTION ──
def predict_winner(team1, team2):
    rank1 = df_matches[df_matches['team1'] == team1]['team1_fifa_rank'].values
    rank2 = df_matches[df_matches['team2'] == team2]['team2_fifa_rank'].values
    if len(rank1) == 0:
        rank1 = df_matches[df_matches['team2'] == team1]['team2_fifa_rank'].values
    if len(rank2) == 0:
        rank2 = df_matches[df_matches['team1'] == team2]['team1_fifa_rank'].values
    rank1 = rank1[0] if len(rank1) > 0 else 50
    rank2 = rank2[0] if len(rank2) > 0 else 50

    s1 = team_stats[team_stats['team'] == team1]
    s2 = team_stats[team_stats['team'] == team2]

    wr1 = s1['win_rate'].values[0] if len(s1) > 0 else 0.35
    wr2 = s2['win_rate'].values[0] if len(s2) > 0 else 0.35
    xg1 = s1['avg_xg'].values[0] if len(s1) > 0 else 0.015
    xg2 = s2['avg_xg'].values[0] if len(s2) > 0 else 0.015

    score1 = (wr1 * 0.4) + (xg1 * 10 * 0.3) + ((1/rank1) * 100 * 0.3)
    score2 = (wr2 * 0.4) + (xg2 * 10 * 0.3) + ((1/rank2) * 100 * 0.3)
    return team1 if score1 >= score2 else team2

def run_round(teams, round_name):
    print(f"\n=== {round_name} ===")
    winners = []
    for i in range(0, len(teams), 2):
        t1, t2 = teams[i], teams[i+1]
        winner = predict_winner(t1, t2)
        print(f"{t1} vs {t2} → 🏆 {winner}")
        winners.append(winner)
    return winners

# ── KNOCKOUT STAGE ──
r16_teams = [
    'Czechia', 'Canada',
    'Brazil', 'Morocco',
    'USA', 'Germany',
    'Japan', 'Belgium',
    'France', 'Spain',
    'Portugal', 'Colombia',
    'England', 'Austria',
    'Iran', 'Türkiye'
]

qf = run_round(r16_teams, "ROUND OF 16")
sf = run_round(qf, "QUARTER FINALS")
final = run_round(sf, "SEMI FINALS")
champion = run_round(final, "FINAL")

print(f"\n🌍 FIFA WORLD CUP 2026 WINNER: {champion[0]} 🏆")

Saving fifa_world_cup_2026_player_performance.csv to fifa_world_cup_2026_player_performance.csv
Saving wc_2026_fixtures.csv to wc_2026_fixtures.csv

=== ROUND OF 16 ===
Czechia vs Canada → 🏆 Canada
Brazil vs Morocco → 🏆 Brazil
USA vs Germany → 🏆 Germany
Japan vs Belgium → 🏆 Belgium
France vs Spain → 🏆 France
Portugal vs Colombia → 🏆 Portugal
England vs Austria → 🏆 England
Iran vs Türkiye → 🏆 Iran

=== QUARTER FINALS ===
Canada vs Brazil → 🏆 Brazil
Germany vs Belgium → 🏆 Belgium
France vs Portugal → 🏆 France
England vs Iran → 🏆 England

=== SEMI FINALS ===
Brazil vs Belgium → 🏆 Brazil
France vs England → 🏆 France

=== FINAL ===
Brazil vs France → 🏆 France

🌍 FIFA WORLD CUP 2026 WINNER: France 🏆
